In [138]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import requests

In [139]:
load_dotenv()

True

In [140]:
model = ChatGroq(model="llama-3.3-70b-versatile") 


In [141]:
class WeatherState(TypedDict):
    user_query: str
    city: str
    weather_api_response: dict
    final_output: str

In [142]:
graph = StateGraph(WeatherState)

In [143]:
def extract_data(state: WeatherState) -> WeatherState:
    user_input = state['user_query']
    prompt = f'You have been given this user query and we are building a weather agent. From this string {user_input}, return only the city name.'
    content = model.invoke(prompt).content
    state['city'] = content
    return state

In [144]:
def call_weather_api(state: WeatherState) -> WeatherState:
    city = state['city']
    api_key = '4753bbf1a1c715a1440c52412a447bcc'
    my_str = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric'
    response = requests.get(my_str)
    if response.status_code == 200:
        data = response.json()
    else:
        data = {'error': f'API returned {response.status_code}'}
    state['weather_api_response'] = data
    return state

In [145]:
def format_output(state: WeatherState) -> WeatherState:
    raw = state['weather_api_response']
    prompt = f'You have been given this raw data {raw} from the weather api key and now your work is that to make it a perfect and format sentence and format it'
    content = model.invoke(prompt).content
    state['final_output'] = content
    return state

In [146]:
graph.add_node('extract_data', extract_data)
graph.add_node('call_weather_api', call_weather_api)
graph.add_node('format_output', format_output)

graph.add_edge(START, 'extract_data')
graph.add_edge('extract_data', 'call_weather_api')
graph.add_edge('call_weather_api', 'format_output')
graph.add_edge('format_output', END)

workflow = graph.compile()

In [147]:
initial_input = {'user_query': 'What is the weather in Peshawar?'}
output = workflow.invoke(initial_input)
print(output['final_output'])

Based on the provided raw data from the weather API, here is a perfectly formatted sentence:

**Current Weather in Peshawar, Pakistan:**
The current temperature in Peshawar, Pakistan is **35.15°C** (feeling like **36.9°C**), with a clear sky and **0%** cloud cover. The humidity is **38%**, and the wind is blowing at a speed of **0.17 m/s** from a direction of **28°**, with a gust of **0.61 m/s**. The visibility is **10 km**, and the atmospheric pressure is **1000 hPa**. The sunrise was at **(timestamp: 1786062548)**, and the sunset will be at **(timestamp: 1786111794)**.

Note: The timestamps for sunrise and sunset are in Unix time format, which can be converted to a human-readable format using a Unix time converter.

Here is a breakdown of the data in a formatted table:

| **Category** | **Value** |
| --- | --- |
| Location | Peshawar, Pakistan |
| Temperature | 35.15°C (feeling like 36.9°C) |
| Cloud Cover | 0% |
| Humidity | 38% |
| Wind Speed | 0.17 m/s |
| Wind Direction | 28° |
|